In [ ]:
from datasets import load_dataset

ds=load_dataset("rajpurkar/squad")
ds.shape

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

{'train': (87599, 5), 'validation': (10570, 5)}

In [ ]:
train_dataset=ds["train"]
train_dataset=train_dataset.shuffle(seed=42).select(range(2000))
val_dataset=ds["validation"]
val_dataset=val_dataset.shuffle(seed=42).select(range(1000))
train_dataset.column_names

['id', 'title', 'context', 'question', 'answers']

In [ ]:
def train_dataset_process(data):
  input=("### Context:\n"
    + data["context"]
    + "\n\n### Question:\n"
    + data["question"]
    + "\n\n### Answer:\n"
    + data["answers"]["text"][0])
  return {"input":input}

train_dataset=train_dataset.map(train_dataset_process,remove_columns=train_dataset.column_names)
print(train_dataset[0])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'input': '### Context:\nThe Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.\n\n### Question:\nWhat percentage of Egyptians polled support death penalty for those leaving Islam?\n\n### Answer:\n84%'}


In [ ]:
train_dataset

Dataset({
    features: ['input'],
    num_rows: 2000
})

In [ ]:
dummy_df=val_dataset.train_test_split(test_size=0.4,seed=42)
test_ds=dummy_df["test"]
val_ds=dummy_df["train"]
test_ds

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 400
})

In [ ]:
val_ds

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 600
})

In [ ]:
val_ds=val_ds.map(train_dataset_process,remove_columns=val_ds.column_names)
val_ds

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Dataset({
    features: ['input'],
    num_rows: 600
})

In [ ]:
def test_dataset_process(data):
  input=("### Context:\n"
    + data["context"]
    + "\n\n### Question:\n"
    + data["question"]
    + "\n\n### Answer:\n")
  output=data["answers"]["text"][0]
  return {"input":input, "output":output}

test_ds=test_ds.map(test_dataset_process,remove_columns=test_ds.column_names)
test_ds

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output'],
    num_rows: 400
})

In [ ]:
import torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz:{device}")

Kullanılan Cihaz:cuda


In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer

model_name="Qwen/Qwen2.5-3B-Instruct"
model=AutoModelForCausalLM.from_pretrained(model_name)
tokenizer=AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
def tokenize(batch):
  return tokenizer(batch["input"],truncation=True,max_length=256)

train_dataset=train_dataset.map(tokenize)
val_ds=val_ds.map(tokenize)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

lora_config=LoraConfig(
    task_type=TaskType.QUESTION_ANS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj","v_proj"]
)

model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


In [ ]:
from transformers import TrainingArguments, Trainer

training_args=TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/lora_finetuning_model",
    learning_rate=1e-3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    fp16=True,
    optim="adamw_torch",
    report_to="none",
    save_strategy="no"
)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
import time

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Eğitim Başladı.")
start_time=time.time()
trainer.train()
end_time=time.time()
print(f"Eğitim Tamamlandı. Eğitim Süresi:{(end_time-start_time)/60}")
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/lora_finetuning_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/lora_finetuning_model")
print("Model Kaydedildi.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Eğitim Başladı.


Step,Training Loss


Eğitim Tamamlandı. Eğitim Süresi:10.032349975903829
Model Kaydedildi.


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM,AutoTokenizer
from bert_score import score

EM_metric=0
sim_metric=0
predictions=[]
references=[]

saved_model_dir="/content/drive/MyDrive/Colab Notebooks/lora_finetuning_model"

model_name="Qwen/Qwen2.5-3B-Instruct"
model=AutoModelForCausalLM.from_pretrained(model_name).to(device)
model=PeftModel.from_pretrained(model,saved_model_dir)
tokenizer=AutoTokenizer.from_pretrained(model_name)

test_length=len(test_ds)

for i in range(test_length):
  data=test_ds[i]["input"]
  inputs=tokenizer(data,return_tensors="pt").to(device)
  true_ans=test_ds[i]["output"].lower().strip()
  with torch.no_grad():
    outputs=model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    outputs=tokenizer.decode(outputs[0],skip_special_tokens=True)
    model_output = outputs[len(data):].strip().lower()

    references.append(true_ans)
    predictions.append(model_output)

    if true_ans==model_output:
      EM_metric+=1
    if true_ans in model_output:
      sim_metric+=1

precision,recall,f1_score=score(predictions,references,lang="en",verbose=True)
print("Bert Score Sonuçları:")
print(f"Precision (Kesinlik): {precision[0]}")
print(f"Recall (Duyarlılık): {recall[0]}")
print(f"F1 Score: {f1_score[0]}")
print("\n")
exact_matches=100*(EM_metric/test_length)
print(f"Exact Matches (EM): {exact_matches}")
sim_metrics=100*(sim_metric/test_length)
print(f"Similar Answers: {sim_metrics}")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (5 outputs left out)

  0%|          | 0/11 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/7 [00:00<?, ?it/s]

done in 2.58 seconds, 155.07 sentences/sec
Bert Score Sonuçları:
Precision (Kesinlik): 1.000000238418579
Recall (Duyarlılık): 1.000000238418579
F1 Score: 1.000000238418579


Exact Matches (EM): 30.25
Similar Answers: 79.75


In [ ]:
import pandas as pd

results_data = []

for i in range(test_length):
    results_data.append({
        "sample_id": i,
        "condition": "LoRA-Fine-Tuning",
        "prediction": predictions[i],
        "reference_answer": references[i]
    })

df_results = pd.DataFrame(results_data)

df_results.to_json("finetuning_results.json", orient="records", indent=4)

In [ ]:
print("Tahmin sayısı:", len(predictions))
print("Referans sayısı:", len(references))
print("Tahminler:", predictions[20:25])
print("Referanslar:", references[20:25])
print("Boş tahmin sayısı:", sum(not p.strip() for p in predictions))

Tahmin sayısı: 400
Referans sayısı: 400
Tahminler: ['the slopes full of vines, settlements crowded on the narrow river banks and scores of castles lined up along the top of the steep slopes, the middle rhine valley can be considered the epitome of the rhine romanticism.', 'pedagogy', 'carbohydrates', 'alpha repertory television service (arts)', 'marquis de la jonquière,']
Referanslar: ['middle rhine valley', 'pedagogy', 'carbohydrates', 'alpha repertory television service (arts)', 'marquis de la jonquière']
Boş tahmin sayısı: 0


In [ ]:
!pip install bert-score

In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 16.6 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
